# Reproducible Analysis Pipeline: Accessibility and Usability Study

**Goal**: Execute the full analysis pipeline from raw data to final figures.
**Prerequisites**: Raw data files must exist in `data/raw/` with valid checksums (generated by T019).
**Environment**: CPU-only (scipy, pandas, matplotlib).

In [ ]:
# Setup: Ensure project root is in path and configure logging
import sys
from pathlib import Path

# Add project root to path if running from notebook directory
project_root = Path.cwd().parent  # Assuming notebook is in code/
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.logger import setup_logger, get_logger
from config.settings import get_settings

# Initialize logging
setup_logger()
logger = get_logger(__name__)

# Load settings
settings = get_settings()
logger.info(f"Project root: {settings.project_root}")
logger.info(f"Data raw path: {settings.data_raw}")
logger.info(f"Data processed path: {settings.data_processed}")

## Step 1: Data Loading and Integrity Verification

Load raw session data from `data/raw/`. The system automatically resolves checksummed files defined in T019.

In [ ]:
import json
import glob
from pathlib import Path
from utils.checksum import verify_checksums_in_directory

raw_data_path = Path(settings.data_raw)
json_files = list(raw_data_path.glob("session_*.json"))

logger.info(f"Found {len(json_files)} raw session files.")

# Verify checksums
if len(json_files) > 0:
    valid_files = verify_checksums_in_directory(raw_data_path)
    logger.info(f"Verified {len(valid_files)} files with valid checksums.")
else:
    logger.warning("No raw data files found. Please run the simulator first.")
    valid_files = []

# Load data into a list of dictionaries
raw_sessions = []
for f in valid_files:
    try:
        with open(f, 'r') as file:
            data = json.load(file)
            raw_sessions.append(data)
    except Exception as e:
        logger.error(f"Failed to load {f}: {e}")

logger.info(f"Loaded {len(raw_sessions)} valid sessions.")

## Step 2: Data Cleaning

Filter incomplete sessions and apply SUS imputation logic (EC-001).

In [ ]:
from analysis.data_cleaner import DataCleaner

cleaner = DataCleaner()
cleaned_df = cleaner.clean(raw_sessions)

logger.info(f"Cleaned dataset shape: {cleaned_df.shape}")
logger.info(f"Columns: {list(cleaned_df.columns)}")

# Display first few rows
cleaned_df.head()

## Step 3: Statistical Analysis

Perform Shapiro-Wilk, Repeated Measures ANOVA, and Holm-Bonferroni correction.

In [ ]:
from analysis.stat_utils import StatUtils

stat_utils = StatUtils()

# Define metrics to test
metrics_to_test = ['completion_time', 'error_count', 'sus_score']

results = {}
for metric in metrics_to_test:
    if metric in cleaned_df.columns:
        logger.info(f"Analyzing {metric}...")
        result = stat_utils.analyze_metric(cleaned_df, metric)
        results[metric] = result
        logger.info(f"  F-stat: {result.get('f_stat'):.4f}, p-value: {result.get('p_value'):.4f}")
    else:
        logger.warning(f"Metric {metric} not found in cleaned data.")

# Descriptive stats for explanation_engagement_time (descriptive only)
if 'explanation_engagement_time' in cleaned_df.columns:
    desc_stats = cleaned_df.groupby('interface_type')['explanation_engagement_time'].agg(['mean', 'std']).reset_index()
    logger.info("Descriptive stats for explanation_engagement_time:")
    logger.info(desc_stats.to_string())

## Step 4: Generate Metrics Summary

Output the summary CSV with F-statistics, p-values, and effect sizes.

In [ ]:
from analysis.generate_metrics_summary import generate_metrics_summary

summary_df = generate_metrics_summary(results, cleaned_df)

output_path = Path(settings.data_processed) / "metrics_summary.csv"
summary_df.to_csv(output_path, index=False)
logger.info(f"Metrics summary saved to {output_path}")

summary_df

## Step 5: Visualization

Generate publication-quality box plots with error bars.

In [ ]:
from analysis.visualizer import Visualizer

visualizer = Visualizer()

# Generate plots for each metric
for metric in metrics_to_test:
    if metric in cleaned_df.columns:
        fig_path = Path(settings.data_processed) / f"plot_{metric}.png"
        visualizer.plot_metric(cleaned_df, metric, output_path=str(fig_path))
        logger.info(f"Saved plot for {metric} to {fig_path}")

# Display the last plot if running in an environment that supports it
import matplotlib.pyplot as plt
if len(metrics_to_test) > 0:
    plt.show()

## Step 6: Generate Summary Report

Create a text report with N achieved, power flags, and exclusion reasons.

In [ ]:
import pandas as pd
from pathlib import Path

report_path = Path(settings.data_processed) / "report_summary.txt"

with open(report_path, 'w') as f:
    f.write("=== Study Analysis Summary Report ===\n\n")
    f.write(f"Total sessions loaded: {len(raw_sessions)}\n")
    f.write(f"Valid sessions analyzed: {len(cleaned_df)}\n")
    f.write(f"Excluded sessions: {len(raw_sessions) - len(cleaned_df)}\n\n")
    
    # Power limitation flags (simple heuristic: if N < 30 per group)
    counts = cleaned_df['interface_type'].value_counts()
    f.write("Power Limitation Flags:\n")
    for iface, count in counts.items():
        if count < 30:
            f.write(f"  - {iface}: N={count} (Underpowered, recommended N>=30)\n")
        else:
            f.write(f"  - {iface}: N={count} (Adequately powered)\n")
    f.write("\n")
    
    f.write("Key Findings:\n")
    for metric, res in results.items():
        sig = "Significant" if res.get('p_value', 1.0) < 0.05 else "Not Significant"
        f.write(f"  - {metric}: {sig} (p={res.get('p_value', 0):.4f})\n")
    
    f.write("\n=== End of Report ===")

logger.info(f"Report saved to {report_path}")

# Print report content
with open(report_path, 'r') as f:
    print(f.read())